In [ ]:
using Pkg
Pkg.activate("..")
using Revise

In [ ]:
using bslLD, CairoMakie, Statistics
using FFTW, DSP
bslLD.greet()

# bslLD.use_cuda!()

In [ ]:
grid =  bslLD.Grid([0.0,-4.0,-4.0],[60.0,4.0,4.0],[32,33,33],0.02,10000,1)

# initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)

f = bslLD.Distribution(grid, 0.1,initFuncv=initFuncv);
e = bslLD.empty_vectorfield(grid);

In [ ]:
function stepSelfConsitent(f,grid)
    bslLD.advectX!(f,grid)
    rho = bslLD.compute_density(f,grid)
    solution = bslLD.solve_fields(bslLD.Moments(rho), grid, bslLD.AdiabaticFieldSolver())
    bslLD.advectV!(f,grid,solution.E)
end

In [ ]:
rhodiag = []
fdiag = []


for i in grid.itime
    grid.index[1] = i
    stepSelfConsitent(f,grid)
    if i%1==0
        push!(fdiag,f.data)
        push!(rhodiag, bslLD.compute_density(f,grid).data)
    end
end


In [ ]:
locData = hcat(map(x-> x.-mean(x), Array.(rhodiag))...)


Nx, Ny = size(locData)
w = kaiser(Ny, 3)

windowed = locData .* w'        # broadcast along second dim (1 × Ny)

heatmap(log.(abs.(fft(windowed)[1:round(Int,Nx/2), 1:100])))